In [ ]:
# 1. Install / check GPU  (Settings -> Accelerator -> GPU T4). TensorFlow is preinstalled; do not reinstall it.
!nvidia-smi
import tensorflow as tf, keras, numpy, sklearn
gpus = tf.config.list_physical_devices('GPU')
print('tensorflow', tf.__version__, '| keras', keras.__version__, '| numpy', numpy.__version__, '| sklearn', sklearn.__version__)
print('GPUs:', [tf.config.experimental.get_device_details(g).get('device_name') for g in gpus])
assert gpus, 'No GPU visible to TensorFlow. Turn on the GPU accelerator and restart.'

In [ ]:
# 2. Unzip code (kept outside /kaggle/working so it is not part of the results)
import pathlib, shutil, zipfile
CODE = pathlib.Path('/kaggle/temp/neurocardio')
shutil.rmtree(CODE, ignore_errors=True); CODE.mkdir(parents=True)
zips = sorted(pathlib.Path('/kaggle/input').rglob('neurocardio_kaggle.zip'))
if zips:
    zipfile.ZipFile(zips[0]).extractall(CODE)
else:   # Kaggle usually unpacks uploaded zips: copy the unpacked tree
    marker = sorted(pathlib.Path('/kaggle/input').rglob('backend/scripts/experiments.py'))
    assert marker, 'neurocardio_kaggle.zip not found under /kaggle/input'
    root = marker[0].parents[2]
    for part in ('backend', 'kaggle', 'docs'):
        shutil.copytree(root / part, CODE / part)
data = sorted(pathlib.Path('/kaggle/input').rglob('neurocardio_dev.npz'))
assert data, 'neurocardio_dev.npz not found under /kaggle/input'
assert not list(pathlib.Path('/kaggle/input').rglob('neurocardio_test_SEALED.npz')), 'Detach the SEALED test dataset: stages 1-3 must not see it.'
print('code:', CODE, '| data:', data[0])

In [ ]:
# 3. Smoke test: every code path on tiny runs. Results are meaningless; it must end with SMOKE OK.
!bash /kaggle/temp/neurocardio/kaggle/run_all.sh --smoke

In [ ]:
# 4. Stage run. Set STAGE to 1, 2 or 3 (one Kaggle session per stage). Re-running resumes.
STAGE = 1
!bash /kaggle/temp/neurocardio/kaggle/stage{STAGE}.sh

In [ ]:
# 5. Zip results for download
import pathlib, zipfile
W = pathlib.Path('/kaggle/working')
out = W / f'neurocardio_results_stage{STAGE}.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in W.rglob('*'):
        if p.is_file() and p.suffix != '.zip':
            z.write(p, p.relative_to(W))
print(out, f'{out.stat().st_size / 1e6:.1f} MB')